In [20]:
import torch 


In [21]:
torch.__version__

'2.13.0+cu130'

In [22]:
from torch import nn
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader

import torchvision
from torchvision.transforms import transforms
from torchvision.datasets import ImageFolder

In [23]:
training_trans=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])
testing_trans=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [24]:
training_trans

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
)

In [25]:
training_data=ImageFolder(root=r"E:\Datasets\plant_classification\train\train",transform=training_trans)
testing_data=ImageFolder(root=r"E:\Datasets\plant_classification\test\test",transform=testing_trans)

In [26]:
im,l=training_data[0]


In [27]:
train_data_loader=DataLoader(dataset=training_data,batch_size=64,shuffle=True)
test_data_loader=DataLoader(dataset=testing_data,batch_size=64,shuffle=True)

In [28]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device 

device(type='cuda')

In [29]:
classes=training_data.classes
classes

['Alstonia Scholaris',
 'Arjun',
 'Bael',
 'Basil',
 'Chinar',
 'Gauva',
 'Jamun',
 'Jatropha',
 'Lemon',
 'Mango',
 'Pomegranate',
 'Pongamia Pinnata']

In [30]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=3,out_channels=12,kernel_size=3)
        self.conv2=nn.Conv2d(in_channels=12,out_channels=24,kernel_size=3)
        self.conv3=nn.Conv2d(in_channels=24,out_channels=28,kernel_size=3)
        self.pool=nn.MaxPool2d(kernel_size=3,stride=3)
        self.relu=nn.ReLU(inplace=True)

        self.fc1=nn.Linear(in_features=28*7*7,out_features=1513)
        self.fc2=nn.Linear(in_features=1513,out_features=800)
        self.fc3=nn.Linear(in_features=800,out_features=400)
        self.fc4=nn.Linear(in_features=400,out_features=12)
    def forward(self,x):
        x=self.pool(self.relu(self.conv1(x)))
        x=self.pool(self.relu(self.conv2(x)))
        x=self.pool(self.relu(self.conv3(x)))
        x=torch.flatten(x,1)
        return (self.fc4(self.relu(self.fc3(self.relu(self.fc2(self.relu(self.fc1(x))))))))
        

In [31]:
model=CNN().to(device=device)
model

CNN(
  (conv1): Conv2d(3, 12, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(12, 24, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(24, 28, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU(inplace=True)
  (fc1): Linear(in_features=1372, out_features=1513, bias=True)
  (fc2): Linear(in_features=1513, out_features=800, bias=True)
  (fc3): Linear(in_features=800, out_features=400, bias=True)
  (fc4): Linear(in_features=400, out_features=12, bias=True)
)

In [32]:
optimizer=Adam(params=model.parameters(),lr=0.001)
loss_func=CrossEntropyLoss()

In [33]:
model.train()
for j in range(25):
    for i,(img,label) in enumerate(train_data_loader):
        img,label=img.to(device),label.to(device)
        y_pred=model(img)
        loss=loss_func(y_pred,label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"loss for {i} is {loss.item()}")

loss for 0 is 2.477431297302246
loss for 1 is 2.467038154602051
loss for 2 is 2.455350637435913
loss for 3 is 2.424168348312378
loss for 4 is 2.4183855056762695
loss for 5 is 2.469846725463867
loss for 6 is 2.3719422817230225
loss for 7 is 2.3727293014526367
loss for 8 is 2.358940362930298
loss for 9 is 2.3434548377990723
loss for 10 is 2.386845588684082
loss for 11 is 2.2595863342285156
loss for 12 is 2.330758810043335
loss for 13 is 2.457913398742676
loss for 14 is 2.4997949600219727
loss for 15 is 2.448215961456299
loss for 16 is 2.391158103942871
loss for 17 is 2.394486427307129
loss for 18 is 2.391838550567627
loss for 19 is 2.402784585952759
loss for 20 is 2.3895421028137207
loss for 21 is 2.379978895187378
loss for 22 is 2.40861177444458
loss for 23 is 2.352299213409424
loss for 24 is 2.2641196250915527
loss for 25 is 2.245108127593994
loss for 26 is 2.2757391929626465
loss for 27 is 2.2227096557617188
loss for 28 is 2.286674976348877
loss for 29 is 2.217759132385254
loss for 30

In [ ]:
model.eval()
correct = 0
total = len(test_data_loader.dataset)

with torch.no_grad():
    for images, labels in test_data_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()

accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")


Accuracy: 0.9862


In [45]:
from PIL import Image
model.load_state_dict(torch.load("model_weights.pth",map_location=device))



# 2. Load image from path
img_path = r"E:\Datasets\plant_classification\valid\valid\Jatropha\0006_0008.JPG"
image = Image.open(img_path).convert("RGB")

# 3. Apply transform
image = testing_trans(image)

# 4. Add batch dimension
image = image.unsqueeze(0).to(device)

# 5. Run inference
model.eval()
with torch.no_grad():
    outputs = model(image)
    preds = torch.argmax(outputs, dim=1)
    prob=torch.softmax(outputs,1)

print("Predicted class index:", classes[preds.item()])
print(preds)
print(f"probability: {torch.max(prob)*100}")
outputs
print(classes)


Predicted class index: Jatropha
tensor([7], device='cuda:0')
probability: 52.22178649902344
['Alstonia Scholaris', 'Arjun', 'Bael', 'Basil', 'Chinar', 'Gauva', 'Jamun', 'Jatropha', 'Lemon', 'Mango', 'Pomegranate', 'Pongamia Pinnata']


In [36]:
torch.save(f="model_weights.pth",obj=model.state_dict())